# **EDA on NYC Housing Market**

In [ ]:
from google.colab import drive
import os

## Mount Google Drive
drive.mount('/content/drive')

## Set the directory path
directory = 'Data Science/Project/Datasets'

os.chdir('/content/drive/MyDrive/' + directory)

In [ ]:
!ls

In [ ]:
import pandas as pd
import numpy as np

bronx_df = pd.read_excel('bronx_2021.xlsx', skiprows=4)
brooklyn_df = pd.read_excel('brooklyn_2021.xlsx', skiprows=4)
manhat_df = pd.read_excel('manhat_2021.xlsx', skiprows=4)
queens_df = pd.read_excel('queens_2021.xlsx', skiprows=4)
staten_df = pd.read_excel('staten_2021.xlsx', skiprows=4)
bronx_df.head()

In [ ]:
total_df = pd.concat([bronx_df, brooklyn_df, manhat_df, queens_df, staten_df])
total_df.info()

# **Data Cleaning**

In [ ]:

district_map = {
    1: "Manhattan",
    2: "Bronx",
    3: "Brooklyn",
    4: "Queens",
    5: "Staten Island",
}

total_df['BOROUGH'] = total_df['BOROUGH'].map(district_map)

In [ ]:
total_df['YEAR'] = pd.DatetimeIndex(total_df['SALE DATE']).year

total_df = total_df[total_df['SALE PRICE'] > 100]
total_df = total_df[total_df['SALE PRICE'] < 10000000 ]
total_df = total_df[total_df['GROSS SQUARE FEET'] > 10]
total_df = total_df[total_df['GROSS SQUARE FEET'] < 100000]
total_df = total_df[total_df['LAND SQUARE FEET'] > 5]
total_df = total_df[total_df['LAND SQUARE FEET'] < 25000]

total_df = total_df[total_df['TOTAL UNITS'] > 0]
total_df = total_df[total_df['YEAR BUILT'] != 0]
total_df = total_df[total_df['COMMERCIAL UNITS'] < 40]
total_df = total_df[total_df['RESIDENTIAL UNITS'] < 500]
total_df = total_df[total_df['YEAR BUILT'] > 1875]

In [ ]:
total_df['SALE PRICE LOG'] = np.log1p(total_df['SALE PRICE'])
total_df['GROSS SQUARE FEET LOG'] = np.log1p(total_df['GROSS SQUARE FEET'])
total_df['LAND SQUARE FEET LOG'] = np.log1p(total_df['LAND SQUARE FEET'])

log_df = total_df[['SALE PRICE LOG', 'GROSS SQUARE FEET LOG', 'LAND SQUARE FEET LOG','RESIDENTIAL UNITS', 'COMMERCIAL UNITS','YEAR BUILT','ZIP CODE', 'YEAR']]
log_df = log_df[log_df['SALE PRICE LOG'] != 0]
log_df = log_df[log_df['GROSS SQUARE FEET LOG'] != 0]
log_df = log_df[log_df['GROSS SQUARE FEET LOG'] > 4]
log_df= log_df[log_df['LAND SQUARE FEET LOG'] != 0]


# **Exploratory Data Analysis**

In [ ]:
total_df.describe()

In [ ]:
print(total_df.columns)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


houses_counting = total_df['BOROUGH'].value_counts()

plt.bar(houses_counting.index, houses_counting.values)
plt.xlabel('City in NY')
plt.ylabel('Number of buildings')
plt.title('Number of Buildings vs District')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(
    data=total_df,
    x='SALE PRICE',
    bins=100,
    kde=True,
)
plt.title('Distribution of Sale Price', fontsize=16)
plt.xlabel('Sale Price', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xlim(0, total_df['SALE PRICE'].quantile(0.99))

plt.show()

In [ ]:
sns.histplot(
    data=total_df,
    x='LAND SQUARE FEET',
    bins=100,
    kde=True,
)
plt.title('Distribution of Land Square Feet', fontsize=16)
plt.xlabel('Land Square Feet', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xlim(0, total_df['LAND SQUARE FEET'].quantile(0.99))

plt.show()

In [ ]:
sns.histplot(
    data=total_df,
    x='GROSS SQUARE FEET',
    bins=100,
    kde=True,
)
plt.title('Distribution', fontsize=16)
plt.xlabel('Gross Square Feet', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xlim(0, total_df['GROSS SQUARE FEET'].quantile(0.99))

plt.show()

In [ ]:
sns.scatterplot(
    data=total_df,
    x='GROSS SQUARE FEET',
    y='SALE PRICE',
    alpha=0.5,
    color='steelblue',
    
    edgecolor=None
)

plt.title('Relationship between Property Size and Sale Price', fontsize=16)
plt.xlabel('Gross Square Feet', fontsize=12)
plt.ylabel('Sale Price', fontsize=12)
plt.xlim(0, total_df['GROSS SQUARE FEET'].quantile(0.99))
plt.ylim(0, total_df['SALE PRICE'].quantile(0.99))
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(
    x=log_df['GROSS SQUARE FEET LOG'],
    y=log_df['SALE PRICE LOG'],
    alpha=0.5,
)
plt.title('Log-Log Relationship between Size and Sale Price', fontsize=16)
plt.xlabel('log(1 + Gross Square Feet)')
plt.ylabel('log(1 + Sale Price)')
plt.show()

In [ ]:
sns.scatterplot(
    x=log_df['SALE PRICE LOG'],
    y=log_df['GROSS SQUARE FEET LOG'],
    alpha=0.5
)
plt.title('Sale Price Log vs Residential units')
plt.show()

In [ ]:
df_numerical = total_df[['RESIDENTIAL UNITS','COMMERCIAL UNITS','LAND SQUARE FEET', 'GROSS SQUARE FEET', 'YEAR BUILT', 'SALE PRICE', 'YEAR']]
corr_matrix = df_numerical.corr(method='spearman')
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', square=True)
plt.title('Spearman Correlation Heatmap')
plt.show()

corr_matrix_log = log_df.corr()
sns.heatmap(corr_matrix_log, annot=True, cmap='coolwarm', square=True)
plt.title('Log Correlation Heatmap')
plt.show()